In [1]:
%matplotlib inline

import os
import sys
import yaml
import copy

import torch
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('../../../')
sys.path.append('../../../../')

%load_ext autoreload
%autoreload 2
    
from computer_vision.yolov11_pose.utils import DEFAULT_CFG_DICT
from computer_vision.yolov11_pose.parameter_parser import parser
from computer_vision.misc import alpha_bending, instance2mask
from computer_vision.yolov11_pose.cfg import get_cfg
from computer_vision.yolov11_pose.data.build import build_yolo_dataset

In [2]:
from matplotlib import patches
import matplotlib.pyplot as plt

cmap = plt.get_cmap('tab10', 10)
plt.rcParams.update({'font.size'   : 12})

In [4]:
args=parser.parse_args('--deterministic'.split())

task='pose'
if task=='pose':
    data_dirpath='D:/data/ultralytics/coco8-pose'
    data_config='../coco8-pose.yaml'
elif task=='detect':
    data_dirpath='D:/data/ultralytics/coco/images/train2017'
    data_config='../coco.yaml'
elif task=='segment':
    data_dirpath='D:/data/ultralytics/coco8-seg'
    data_config='../coco8-seg.yaml'

hyp=get_cfg()
dataset=build_yolo_dataset(args=args, cfg=hyp, task=task, img_path=data_dirpath, batch=args.batch_size or hyp.batch, 
                   data=data_config,  mode='train', rect=False, stride=32, channels=3)
for i in range(4): print(dataset.labels[i]['cls'].shape)

In data.dataset.YOLODataset.get_labels cache_path D:\data\ultralytics\coco8-pose\labels\train.cache
Scanning D:\data\ultralytics\coco8-pose\labels\train.cache... 8 images, 0 backgrounds, 0 corrupts
(1, 1)
(2, 1)
(3, 1)
(1, 1)


In [ ]:
batch=hyp.batch
workers=1
shuffle=True
drop_last=True
pin_memory=True
# build_dataloader(dataset, batch:int, workers:int, shuffle:bool=True, drop_last:bool=False,
# pin_memory:bool=True)->torch.utils.data.DataLoader
"""Create and return a DataLoader for training and validation
Args:
    dataset (torch.utils.data.Dataset): Dataset to load data from
    batch (int): Batch size for the dataloader
    workers (int): Number of worker threads for loading data
    shuffle (bool, optional): Whether to shuffle the dataset
    drop_last (bool, optional): Whether to drop the last incomplete batch
    pin_memory (bool, optional): Whether to use pinned memory for dataloader
Returns:
    (torch.utils.data.DataLoader): A dataloader that can be used for training and validation
"""
batch=min(batch, len(dataset))
nd=torch.cuda.device_count() # number of CUDA devices
nw=min(os.cpu_count()//max(nd, 1), workers) # number of workers
torch.utils.data.DataLoader(dataset, batch_size=batch, shuffle=shuffle, num_workers=nw,
                           collate_fn=getattr(dataset, 'collate_fn', None),
                           pin_memory=nd>0 and pin_memory, 
                           drop_last=drop_last and len(dataset)%batch!=0,
                           worker_init_fn=seed_worker, )

In [10]:
getattr(dataset, 'collate_fn', None)

In [9]:
DataLoader(dataset, batch_size=1, shuffle=False, sampler=None,
           batch_sampler=None, num_workers=0, collate_fn=None,
           pin_memory=False, drop_last=False, timeout=0,
           worker_init_fn=None, *, prefetch_factor=2,
           persistent_workers=False)

32

In [14]:
batch=[torch.rand(3,100,100),torch.rand(3,100,100),torch.rand(3,100,100)]
torch.stack(batch,0).shape

[autoreload of computer_vision.yolov11_pose.data.dataset failed: Traceback (most recent call last):
  File "C:\Users\Sureerat\miniforge3\envs\op_cv\lib\site-packages\IPython\extensions\autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "C:\Users\Sureerat\miniforge3\envs\op_cv\lib\site-packages\IPython\extensions\autoreload.py", line 475, in superreload
    module = reload(module)
  File "C:\Users\Sureerat\miniforge3\envs\op_cv\lib\importlib\__init__.py", line 169, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 619, in _exec
  File "<frozen importlib._bootstrap_external>", line 879, in exec_module
  File "<frozen importlib._bootstrap_external>", line 1017, in get_code
  File "<frozen importlib._bootstrap_external>", line 947, in source_to_code
  File "<frozen importlib._bootstrap>", line 241, in _call_with_frames_removed
  File "D:\dev\computer_vision\yolov11_pose\notebook\../../..\computer_vision\yolov11_pos

torch.Size([3, 3, 100, 100])

In [2]:
np.array([[0],[1]]).shape

(2, 1)